# Colab Setup — GenAI Assignment 1
Run these cells at the start of **every** Colab session (Runtime → Change runtime type → **T4 GPU** first).

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Drive holds everything that must survive a disconnect: dataset archives, checkpoints, Optuna studies, ONNX exports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DRIVE = '/content/drive/MyDrive/GenAI_A1'
for sub in ['archives', 'checkpoints', 'optuna', 'onnx', 'outputs']:
    os.makedirs(f'{PROJECT_DRIVE}/{sub}', exist_ok=True)

## 3. Get the code from GitHub
Set `REPO_URL` to your repository. For a private repo, add a `GITHUB_TOKEN` secret (🔑 icon in the left sidebar).

In [ ]:
REPO_URL = 'https://github.com/fsdev87/generative-ai-A1.git'
REPO_DIR = '/content/repo'

from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
    REPO_URL = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass  # public repo

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Datasets
First session: downloads archives into Drive (~830 MB total). Later sessions: copies from Drive and extracts onto the fast local disk (`/content/data`). Training reads from local disk, never directly from Drive (Drive is slow for thousands of small files).

In [ ]:
!python scripts/download_data.py --archive-dir {PROJECT_DRIVE}/archives --data-dir /content/data
!ls /content/data /content/data/oxford_pets /content/data/FS2K

## 6. Experiment tracking (Weights & Biases)
Add your W&B API key as a Colab secret named `WANDB_API_KEY` (from https://wandb.ai/authorize).

In [ ]:
import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

## 7. Paths for training scripts
Training scripts should read these env vars so the same code runs locally and on Colab.

In [ ]:
os.environ['DATA_DIR'] = '/content/data'
os.environ['CKPT_DIR'] = f'{PROJECT_DRIVE}/checkpoints'
os.environ['OPTUNA_DIR'] = f'{PROJECT_DRIVE}/optuna'
os.environ['ONNX_DIR'] = f'{PROJECT_DRIVE}/onnx'
os.environ['OUTPUT_DIR'] = f'{PROJECT_DRIVE}/outputs'

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 8. Run training (examples — scripts come later)
```
!python -m src.task1.train_optuna --n-trials 30
!python -m src.task1.train --config configs/task1_best.yaml
```